# Chromatin Toggle — GPU runner (Google Colab)

One-click GPU runs for the multiscale KG-GNN.

**Setup:** `Runtime -> Change runtime type -> Hardware accelerator: T4 GPU -> Save`, then `Runtime -> Run all`.

Colab ships PyTorch+CUDA and the training data (`cross_pathway_eval.csv`, ~8 MB) is committed in the repo, so this is just clone + install + run. Runs that take ~45 min on a loaded CPU finish in ~1-2 min on a T4.

In [ ]:
# 1. Confirm the GPU (expect a Tesla T4). If this errors, enable the GPU runtime (see above).
!nvidia-smi

## 2. Get the code + data

This repo is **private**, so two things need authorization:

**(a) Opening this notebook** — you're doing that now (you uploaded it, or authorized Colab's GitHub access for private repos).

**(b) Cloning the repo below** — needs a read-only GitHub token:
1. Make a fine-grained token: GitHub → Settings → Developer settings → **Fine-grained tokens** → repo `nlipieta/MultiscaleProject`, **Contents: Read-only**.
2. In Colab, click the **🔑 key icon** (left sidebar) → **Add new secret** → name `GH_TOKEN`, paste the value, enable "Notebook access".
3. Run the next cell. (Never paste the token into a cell — the secret keeps it out of the saved notebook.)

In [ ]:
import os

# --- PRIVATE repo (default): reads GH_TOKEN from Colab Secrets (step 2 above) ---
from google.colab import userdata
tok = userdata.get('GH_TOKEN')
if not os.path.isdir('MultiscaleProject'):
    !git clone https://{tok}@github.com/nlipieta/MultiscaleProject.git

# --- if you ever make the repo PUBLIC, use this instead (no token needed) ---
# if not os.path.isdir('MultiscaleProject'):
#     !git clone https://github.com/nlipieta/MultiscaleProject.git

%cd MultiscaleProject

In [ ]:
# 3. Install. Colab already has a CUDA-enabled PyTorch, so --no-deps avoids downgrading it.
!pip install -q pyyaml pandas scikit-learn anndata
!pip install -q -e . --no-deps

import torch
print('CUDA available:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

## 4. Main experiment — baseline comparison on GPU

KG-GNN vs majority / logistic regression / random forest / gradient boosting, honest grouped-split CV, converged training. `--device cuda` uses the T4.

In [ ]:
# Beefed-up, fully-converged config (free on a GPU). Reports overall acc / balanced acc /\n# program recall / macro-F1 / program-AUPRC. This shows the model's TRUE converged ceiling.\n!python -m chromatin_toggle.baselines --group-split --class-weight \\\n    --models majority logreg rforest gboost \\\n    --kfolds 5 --subsample 8000 --steps 8 --hidden 128 --epochs 120 \\\n    --seed 0 --device cuda

## 5. Scaling-law vs capacity — the data-vs-capacity test

This directly tests "does the model keep improving with more data / capacity?". On CPU the earlier scaling run plateaued at ~16k cells; re-running with a bigger, fully-converged model on GPU shows whether that plateau is a **data** ceiling or a **capacity** ceiling. Bump `--hidden` (e.g. 64 -> 128 -> 256) across runs to probe capacity.

In [ ]:
# Scaling vs capacity: does the plateau lift with more data AND a bigger model?\n!python -m chromatin_toggle.scaling --data data/cross_pathway_eval.csv \\\n    --sizes 1000 2000 4000 8000 16000 --epochs 120 --steps 8 --hidden 128 \\\n    --seeds 0 1 2 --device cuda

## 6. (Optional) multi-seed baseline for significance

Now that it's fast, run seeds 0/1/2 for real error bars on the KG-GNN-vs-logreg comparison.

In [ ]:
for seed in [0, 1, 2]:\n    print(f'\\n================ seed {seed} ================')\n    !python -m chromatin_toggle.baselines --group-split --class-weight \\\n        --models majority logreg rforest gboost \\\n        --kfolds 5 --subsample 8000 --steps 8 --hidden 128 --epochs 120 \\\n        --seed {seed} --device cuda

## 7. (Optional) ablation + perturbation on GPU

Which mechanisms are load-bearing, and the hypertrophy perturbation test.

In [ ]:
!python -m chromatin_toggle.ablate --group-split --class-weight \\\n    --subsample 8000 --steps 8 --hidden 128 --epochs 120 --seed 0 --device cuda\nprint('\\n================ perturbation ================')\n!python -m chromatin_toggle.perturb --subsample 8000 --steps 8 --hidden 128 --epochs 120 --seed 0 --device cuda